# C1 · Cosmos3 (MoT: reasoner + 확산 generator) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/C1_cosmos3.ipynb)

원본: diffusers `Cosmos3OmniTransformer` (`src/diffusers/models/transformers/transformer_cosmos3.py`,
main 커밋 `c60830e` 고정 = PyPI `diffusers 0.41.0` 과 같은 파일). 내 구현: `miniarch/models/cosmos3.py`.
가중치 이름은 실제 체크포인트(`nvidia/Cosmos3-*/transformer`)와 같습니다.

Cosmos3는 **층마다 파라미터가 두 벌**인 Mixture-of-Transformers입니다.
- reasoner(이해) 탑: 텍스트 토큰, **causal** attention → 그대로 LLM (lm_head 로 다음 토큰 예측)
- generator(확산) 탑: 영상·소리·행동 latent 토큰, query는 generator, key/value는 **reasoner + generator 전부(양방향)**
- 두 탑은 attention에서만 만나고 정보는 reasoner → generator 한 방향으로만 흐릅니다.
- 위치: interleaved 3D M-RoPE (t, h, w), head_dim 전체 회전. 시간 조건은 noisy 프레임 토큰에 timestep 임베딩을 **더하기만** 함
  (조건 프레임 = 깨끗한 토큰 → 이미지→영상 / 편집 / 행동 조건 생성이 같은 경로)
- 행동: embodiment(domain)마다 가중치가 다른 DomainAwareLinear 입출력

축소 config (모든 경로가 한 번씩 쓰이도록)
- 2층, d=64, head 4 / KV head 2 (GQA), head_dim 16, M-RoPE 채널 [4, 2, 2] (실제 [24, 20, 20] 과 같은 interleave 규칙)
- 텍스트 7 + 영상 15 + 소리 4 + 행동 5 = 31 토큰. 위치는 파이프라인처럼 실수(fps 보정) t 와 (h, w)
- 영상 2개: ① T=2, 3×5 latent (patch 2 의 배수가 아님 → 0 padding 경로), 0번 프레임은 조건(깨끗), 1번만 noisy
  ② T=3, 2×2 latent, 0·2번 프레임만 noisy (띄엄띄엄). 토큰마다 다른 timestep
- 소리 4프레임(전부 noisy), 행동 5스텝(0번은 조건) + domain 2 (domain 3개 중)
- 원본이 0 으로 만드는 modality embed, N(0,1) 인 행동 domain 표는 스케일을 맞춘 랜덤 값으로 채웁니다.
- Edge(Nemotron: relu2 MLP, reasoner QK-norm 없음 + generator 용 reasoner key norm) 변형은 자체 점검에서 한 번 더 비교합니다.

출력 비교는 영상·소리·행동 예측을 이어 붙인 한 줄 텐서, 층별 비교는 reasoner 층 출력과 두 탑의 attention 출력·generator MLP 입니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. 단계 전용 자체 점검 → 5. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip -q install -U -r requirements-colab.txt
# Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
!pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
!pip -q install -U 'diffusers==0.41.0'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("C1")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('nvidia/Cosmos3-Nano', '^layers\\.0\\.|^(action|audio)_', subfolder='transformer')
peek('nvidia/Cosmos3-Edge', '^layers\\.0\\.self_attn\\.', subfolder='transformer')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## 단계 전용 자체 점검

### 한 방향 정보 흐름 · reasoner causal

MoT의 핵심 성질을 내 구현으로 직접 확인합니다.
1. 생성 토큰(영상·소리·행동 latent, timestep)을 바꿔도 reasoner 출력은 **그대로**, generator 출력은 바뀜
2. 뒤쪽 텍스트 토큰을 바꾸면 그 앞 reasoner 출력은 그대로 (causal), 뒤는 바뀌고, generator 출력도 바뀜 (generator 는 reasoner 를 봄)
3. ablation: `gen_sees_und=False` 이면 텍스트를 바꿔도 generator 불변, `und_sees_gen=True` 이면 생성 토큰이 reasoner 를 바꿈

In [ ]:
import dataclasses
from miniarch.models import cosmos3 as C
ref, inp = S.ref()
m = C.verify_pair(ref)["mine"].to(dev).eval()

def run(x, model=m):
    with torch.no_grad():
        o = model(**S.to_device(x, dev))
    return o.und_hidden, o.gen_hidden

def d(a, b):
    return (a - b).abs().max().item()

g_ = torch.Generator().manual_seed(7)
x_gen = dict(inp)
for k in ("vision_tokens", "sound_tokens", "action_tokens"):
    x_gen[k] = [t + torch.randn(t.shape, generator=g_) for t in inp[k]]
x_gen["vision_timesteps"] = inp["vision_timesteps"].flip(0)
x_txt = dict(inp)
cut = 4
ids = inp["input_ids"].clone(); ids[cut:] = (ids[cut:] + 1) % V.cases.VOCAB
x_txt["input_ids"] = ids

u0, g0 = run(inp)
u1, g1 = run(x_gen)
report("생성 토큰 변경 → reasoner 불변", d(u0, u1) < 1e-6, d(u0, u1))
report("생성 토큰 변경 → generator 변함", d(g0, g1) > 1e-3, d(g0, g1))
u2, g2 = run(x_txt)
report(f"텍스트 {cut}번 이후 변경 → 앞 reasoner 불변", d(u0[:cut], u2[:cut]) < 1e-6, d(u0[:cut], u2[:cut]))
report(f"텍스트 {cut}번 이후 변경 → 뒤 reasoner 변함", d(u0[cut:], u2[cut:]) > 1e-3, d(u0[cut:], u2[cut:]))
report("텍스트 변경 → generator 변함 (reasoner 를 봄)", d(g0, g2) > 1e-3, d(g0, g2))

def variant(**sw):
    m2 = C.Cosmos3(dataclasses.replace(m.config, **sw)).to(dev).eval()
    m2.load_state_dict(m.state_dict())
    return m2
m_a = variant(gen_sees_und=False)
_, ga0 = run(inp, m_a); _, ga1 = run(x_txt, m_a)
report("gen_sees_und=False → 텍스트 바꿔도 generator 불변", d(ga0, ga1) < 1e-6, d(ga0, ga1))
m_b = variant(und_sees_gen=True)
ub0, _ = run(inp, m_b); ub1, _ = run(x_gen, m_b)
report("und_sees_gen=True → 생성 토큰이 reasoner 를 바꿈", d(ub0, ub1) > 1e-3, d(ub0, ub1))

### attention map 확인

- `und_probs` [H, N_und, N_und]: 대각선 위(미래)는 정확히 0, 행 합 1
- `gen_probs` [H, N_gen, N_und + N_gen]: 앞쪽 N_und 칸이 reasoner 몫. generator 질의가 텍스트에 주는 평균 확률을 출력합니다.

In [ ]:
from miniarch.models import cosmos3 as C
ref, inp = S.ref()
m = C.verify_pair(ref)["mine"].to(dev).eval()
names = [n for n, _ in m.named_modules() if n.endswith(("und_probs", "gen_probs"))]
with V.capture(m, names) as st, torch.no_grad():
    m(**S.to_device(inp, dev))
nu = inp["und_len"]
for i in range(m.config.n_layer):
    pu, pg = st[f"layers.{i}.self_attn.und_probs"], st[f"layers.{i}.self_attn.gen_probs"]
    fut = pu.triu(1).abs().max().item()
    err = max((pu.sum(-1) - 1).abs().max().item(), (pg.sum(-1) - 1).abs().max().item())
    print(f"layer {i} und={tuple(pu.shape)} gen={tuple(pg.shape)}  generator→텍스트 평균 확률 {pg[..., :nu].sum(-1).mean():.3f}")
    report(f"layer {i} reasoner 미래 확률 = 0", fut == 0, fut)
    report(f"layer {i} 확률 행 합 = 1", err < 1e-5, err)

### Edge 변형 (Nemotron 백본) 원본 비교

Cosmos3-Edge 는 `hidden_act="relu2"` (gate 없는 relu² MLP, Nemotron RMSNorm), reasoner QK-norm 없음,
generator 가 보는 reasoner key 에만 별도 norm(`k_norm_und_for_gen`)을 씁니다. 소리 없이 같은 방식으로 원본과 비교합니다.

In [ ]:
from miniarch.models import cosmos3 as C
V.set_deterministic(0)
r2, x2 = S.ref(edge=True)
V.perturb_params(r2)
pair = C.verify_pair(r2)
mine2 = pair.pop("mine")
rep = V.check_equivalence(r2, mine2, x2, atol=1e-4, device=dev, **pair)
print(rep.summary())
report("Edge 변형 원본 동치", rep.ok, max(r.max_abs for r in rep.results))

### ablation 스위치

부품별 비교 실험에 쓸 스위치를 하나씩 바꾸고 forward 가 정상(유한값)이고 출력이 실제로 바뀌는지 확인합니다.

In [ ]:
import dataclasses
from miniarch.models import cosmos3 as C
ref, inp = S.ref()
m = C.verify_pair(ref)["mine"].to(dev).eval()
x = S.to_device(inp, dev)
with torch.no_grad():
    base = m(**x).flat()
for sw, val in [("und_causal", False), ("gen_sees_und", False), ("und_sees_gen", True), ("share_towers", True),
                ("use_timestep", False), ("mrope_interleaved", False)]:
    m2 = C.Cosmos3(dataclasses.replace(m.config, **{sw: val})).to(dev).eval()
    m2.load_state_dict(m.state_dict())
    with torch.no_grad():
        out = m2(**x).flat()
    dd = (out - base).abs().max().item()
    report(f"{sw}={val} 유한 · 출력 변함", bool(torch.isfinite(out).all()) and dd > 1e-4, dd)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/C1_cosmos3.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_C1_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")